# Stress Prediction — 멀티 시드 앙상블 (구조 변경 없음)

v34 구조를 100% 그대로 유지합니다 (quantile, weight, 피처, 하이퍼파라미터 전부 원본).
유일한 차이: `ExtraTreesRegressor`를 **서로 다른 random_state로 N번 학습**해서 트리 예측을
평균냅니다. `max_features=1, min_samples_leaf=1`은 무작위성이 매우 커서, 시드 하나(42)의
결과가 우연히 나쁠 수도 좋을 수도 있습니다. 여러 시드를 평균내면 그 무작위성(분산)이 줄어듭니다.

이건 오늘 시도한 것들과 성격이 다릅니다 — 편향(모델이 무엇을 학습하는가)을 바꾸는 게 아니라
분산(같은 걸 학습해도 매번 결과가 흔들리는 정도)을 줄이는 거라, 구조적으로 항상 같거나 나은
방향입니다 (더 나빠질 이론적 이유가 없습니다).


In [ ]:

from __future__ import annotations

import os
import time
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold

TARGET = "stress_score"
ID_COLUMN = "ID"
N_ESTIMATORS = 1200
TREE_QUANTILE = 0.54
PAIR_QUANTILE = 0.48
PAIR_WEIGHT = 0.28

SEEDS = [42, 1, 7, 123, 2026, 77, 999, 55]  # v34 원본 시드(42) 포함, 총 8개

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open")
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/cl_6")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COPY_COUNTS = {
    "mean_working": 1, "bmi": 4, "cholesterol": 2, "height": 2,
    "glucose": 3, "weight": 2, "cholesterol_glucose_ratio": 2, "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working", "bmi", "cholesterol", "height",
    "glucose", "weight", "cholesterol_glucose_ratio", "bone_density",
]


def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (result["glucose"] + 1e-6)
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, n_copies in FEATURE_COPY_COUNTS.items():
        for i in range(1, n_copies + 1):
            result[f"{feature}__copy{i}"] = result[feature]
    return result


def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer([
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
        ("numerical", SimpleImputer(strategy="median", add_indicator=True), numerical),
    ])


def tree_prediction_single_seed(train_features, val_features, target, seed) -> np.ndarray:
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    preprocessor = make_preprocessor(weighted_train)
    train_matrix = preprocessor.fit_transform(weighted_train)
    val_matrix = preprocessor.transform(weighted_val)
    model = ExtraTreesRegressor(
        n_estimators=N_ESTIMATORS, min_samples_leaf=1, max_features=1,
        criterion="squared_error", random_state=seed, n_jobs=-1,
    )
    model.fit(train_matrix, target)
    tree_predictions = np.column_stack([t.predict(val_matrix) for t in model.estimators_])
    return np.quantile(tree_predictions, TREE_QUANTILE, axis=1)


def tree_prediction_multiseed(train_features, val_features, target, seeds) -> np.ndarray:
    """여러 시드로 학습한 트리 예측(라운딩 전)을 평균."""
    preds = [tree_prediction_single_seed(train_features, val_features, target, s) for s in seeds]
    return np.mean(preds, axis=0)


def empirical_rank_transform(train_values, test_values):
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_prediction(train_features, val_features, target) -> np.ndarray:
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )
    target_values = target.to_numpy(dtype=float)
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argmin(distance, axis=1)
        neighbor_targets.append(target_values[nearest_idx])
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


## 5-Fold CV — 시드 1개(원본) vs 시드 N개 평균 비교

시드를 하나씩 늘려갈 때마다 CV MAE가 어떻게 바뀌는지 누적으로 보여줍니다
(1개=원본, 2개=42+1 평균, 3개=42+1+7 평균, ...).


In [2]:

train = pd.read_csv(DATA_DIR / "train.csv")
raw_X = add_features(train.drop(columns=[TARGET, ID_COLUMN])).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=42)  # CV 분할 자체는 고정 (여러 실험과 비교 가능하도록)

# fold별로 각 시드의 tree raw 예측(라운딩 전)과 pair 예측을 저장
fold_tree_preds = []  # list of dict{seed: array}
fold_pair_preds = []
fold_y_val = []
fold_val_idx = []

start = time.time()
for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_feat = raw_X.iloc[train_idx].reset_index(drop=True)
    val_feat = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train = y.iloc[train_idx].reset_index(drop=True)
    y_val = y.iloc[val_idx].to_numpy(dtype=float)

    seed_preds = {}
    for seed in SEEDS:
        seed_preds[seed] = tree_prediction_single_seed(train_feat, val_feat, y_train, seed)

    pred_pair = pair_prediction(train_feat, val_feat, y_train)

    fold_tree_preds.append(seed_preds)
    fold_pair_preds.append(pred_pair)
    fold_y_val.append(y_val)
    fold_val_idx.append(val_idx)

    print(f"Fold {fold} done ({time.time() - start:.0f}s 경과)")

print()
print("### 시드 개수 누적 앙상블 CV MAE 비교")
n_total = len(y)
for n_seeds in range(1, len(SEEDS) + 1):
    used_seeds = SEEDS[:n_seeds]
    oof = np.zeros(n_total)
    for seed_preds, pred_pair, val_idx in zip(fold_tree_preds, fold_pair_preds, fold_val_idx):
        tree_avg = np.mean([seed_preds[s] for s in used_seeds], axis=0)
        combined = np.clip(np.round((1 - PAIR_WEIGHT) * tree_avg + PAIR_WEIGHT * pred_pair, 2), 0.0, 1.0)
        oof[val_idx] = combined
    mae = mean_absolute_error(y, oof)
    print(f"시드 {n_seeds}개 ({used_seeds}): CV MAE = {mae:.6f}")

print()
print("(참고) v34 원본(시드 42 하나) CV MAE: 0.148033")


Fold 1 done (13s 경과)
Fold 2 done (27s 경과)
Fold 3 done (41s 경과)
Fold 4 done (55s 경과)
Fold 5 done (69s 경과)

### 시드 개수 누적 앙상블 CV MAE 비교
시드 1개 ([42]): CV MAE = 0.148033
시드 2개 ([42, 1]): CV MAE = 0.147947
시드 3개 ([42, 1, 7]): CV MAE = 0.147973
시드 4개 ([42, 1, 7, 123]): CV MAE = 0.147990
시드 5개 ([42, 1, 7, 123, 2026]): CV MAE = 0.148013
시드 6개 ([42, 1, 7, 123, 2026, 77]): CV MAE = 0.147977
시드 7개 ([42, 1, 7, 123, 2026, 77, 999]): CV MAE = 0.147967
시드 8개 ([42, 1, 7, 123, 2026, 77, 999, 55]): CV MAE = 0.147957

(참고) v34 원본(시드 42 하나) CV MAE: 0.148033


## 최종 제출 파일 생성

가장 낮은 CV MAE를 준 시드 개수로 최종 제출 파일을 만듭니다. 시간이 없으면 SEEDS 전체(8개) 그대로 써도 됩니다
(시드가 많을수록 안정적이지만 학습 시간도 그만큼 늘어남 — 8개면 원본 대비 8배 시간이 걸립니다).


In [4]:

BEST_N_SEEDS = 8  # 위 결과에서 가장 낮은 CV MAE를 준 시드 개수로 수정

test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")
raw_X_test = add_features(test.drop(columns=[ID_COLUMN])).reset_index(drop=True)

final_pair = pair_prediction(raw_X, raw_X_test, y)
final_tree = tree_prediction_multiseed(raw_X, raw_X_test, y, SEEDS[:BEST_N_SEEDS])
final_prediction = np.clip(np.round((1 - PAIR_WEIGHT) * final_tree + PAIR_WEIGHT * final_pair, 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_multiseed_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/submit_multiseed_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.60
1,TEST_0001,0.97
2,TEST_0002,0.19
3,TEST_0003,0.51
4,TEST_0004,0.53
5,TEST_0005,0.46
6,TEST_0006,0.78
7,TEST_0007,0.47
8,TEST_0008,0.58
9,TEST_0009,0.88
